## Read job ad

- open more files
- with once, then just open after. with block 

In [20]:
with open("../data/ads_1.txt", "r") as file, open("../data/ads2.txt", "r") as file2, open("../data/ads3.txt", "r") as file3:
    ads_1 = file.read()
    ads_2 = file2.read()
    ads_3 = file3.read()

ads_2[:100], ads_2[:100], ads_3[:100]

('Engineering Operations Technician , Data Center Engineering Operations\nIdentifiant du poste: 3078595',
 'Engineering Operations Technician , Data Center Engineering Operations\nIdentifiant du poste: 3078595',
 'At NOBA Bank Group, we are unlocking new possibilities as we enter an exciting phase with great unta')

## Model

- Harder rules the agent most follow
- Field makes that

In [21]:
from pydantic import BaseModel, Field

class job(BaseModel):
    job_title:str = Field(description="Make sure its only on job title per job ad")
    description:str = Field(description="Keep it clean, and medium short")
    summary:str = Field(description="Max 200 words per summary")
    responsibilites:str
    words_in_article:int

## Agent

In [23]:
from pydantic_ai import Agent
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

job_ad_agent = Agent(
    "openrouter:openai/gpt-oss-120b:free",
    system_prompt="""You are an job ad summarizer, summarise ad in a good reading language. not to long. 
    You will summarize 3 different job ads
""",    
)

combined = f"Ad 1:\n{ads_1}\n\nAd 2:\n{ads_2}\n\nAd 3:\n{ads_3}"
result = await job_ad_agent.run(combined, output_type=job)
result.output

job(job_title='Data Engineer', description='Data Engineer / Analytics Engineer at Instabee', summary='Instabee’s new Data Platform team seeks a hybrid Data Engineer/Analytics Engineer to shape a modern data stack (FiveTran, Python, DBT, GCP, Snowflake, Tableau). You’ll lead source integrations, build CI/CD pipelines, orchestrate Airflow jobs, and design trustworthy data models, bridging business and tech. The role is Stockholm‑based, hybrid, and offers a vibrant office, flexible hours, parental benefits, and a fun, inclusive culture.', responsibilites='Design and build modern data platform on GCP and Snowflake, create integration patterns for APIs, MongoDB, MySQL, set up CI/CD and IaC, orchestrate pipelines with Airflow, model data using DBT/SQL and Python, collaborate with stakeholders, ensure data discoverability and trust, support analytics projects.', words_in_article=78)

In [25]:
ads = [ads_1, ads_2, ads_3]
results = []

for i, ad in enumerate(ads, start=1):
    result = await job_ad_agent.run(ad, output_type=job)
    results.append(result.output)

# Exportera markdown-filer
for i, r in enumerate(results, start=1):
    with open(f"../data/job_summary_{i}.md", "w") as f:
        f.write(f"{r.job_title}\n\n")
        f.write(f"Description\n{r.description}\n\n")
        f.write(f"Summary\n{r.summary}\n\n")
        f.write(f"Responsibilities\n{r.responsibilites}\n\n")
        f.write(f"Words in article: {r.words_in_article}\n")